In [2]:
# -*- coding: utf-8 -*-
import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# =========================
# 配置
# =========================
fold_dir   = "./best_xgb_5fold_saved"
n_folds    = 5

GRID_SIZE  = 20
MIN_IN_SAMPLES = 30
Q_LOW, Q_HIGH = 0.05, 0.95

k_list       = np.arange(0.05, 0.15, 0.05)
a_list       = [10, 100, 500, 1000, 10000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

LABEL_DIFF_MODE = "jaccard"
K_SWD_CAP = 300

OUT_CSV = os.path.join(fold_dir, "AD_SAL_multilabel_search_VERBOSE_live.csv")

# =========================
# 工具
# =========================
def l2_normalize(X, eps=1e-12):
    X = X.astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def cosine_sim_query(Xq_n, Xtr_n):
    return np.clip(Xq_n @ Xtr_n.T, 0.0, 1.0).astype(np.float32)

def compute_weights(sim_vals, a, epsilon):
    s = np.clip(sim_vals, epsilon, 1.0)
    return np.exp(-a * (1.0 - s) / s).astype(np.float32)

def pairwise_label_diff_scalar(y_t, y_nei, mode="jaccard", eps=1e-12):
    y_t = y_t.astype(np.int8, copy=False)
    y_nei = y_nei.astype(np.int8, copy=False)
    if mode == "hamming":
        return np.mean(np.abs(y_nei - y_t[None, :]), axis=1).astype(np.float32)
    if mode == "jaccard":
        inter = np.sum((y_nei == 1) & (y_t[None, :] == 1), axis=1).astype(np.float32)
        union = np.sum((y_nei == 1) | (y_t[None, :] == 1), axis=1).astype(np.float32)
        j = inter / (union + eps)
        return (1.0 - j).astype(np.float32)
    raise ValueError

def per_label_ap(y_true, y_prob):
    y_true = np.asarray(y_true).astype(np.int8)
    y_prob = np.asarray(y_prob).astype(np.float32)
    L = y_true.shape[1]
    ap = np.full(L, np.nan, dtype=float)
    for j in range(L):
        yt = y_true[:, j]
        if len(np.unique(yt)) < 2:
            continue
        ap[j] = average_precision_score(yt, y_prob[:, j])
    return ap

def macro_auprc_with_baseline_fill(y_true_in, y_prob_in, ap_baseline, label_idx):
    ap_in = per_label_ap(y_true_in, y_prob_in)
    ap_filled = ap_in.copy()
    nan_mask = np.isnan(ap_filled)
    ap_filled[nan_mask] = ap_baseline[nan_mask]
    return float(np.mean(ap_filled[label_idx]))

def load_fold_npz(fold_id: int):
    path = os.path.join(fold_dir, f"fold_{fold_id:02d}.npz")
    z = np.load(path, allow_pickle=True)
    return z["X_tr"], z["y_tr"], z["X_va"], z["y_va"], z["y_prob_va"]

def append_row_to_csv(row: dict, csv_path: str):
    df = pd.DataFrame([row])
    header = not os.path.exists(csv_path)
    df.to_csv(csv_path, mode="a", index=False, encoding="utf-8-sig", header=header)

def compute_SWD_topk_with_progress(Xtr, ytr, k_swd_eff, a, eps, fold_id, print_every=500):
    """
    这里会构造 (N,N) 相似度矩阵 -> 最耗时
    加了进度打印，保证你知道在跑
    """
    Xn = l2_normalize(Xtr)
    sim = cosine_sim_query(Xn, Xn)  # (N,N)

    N = ytr.shape[0]
    SWD = np.zeros(N, dtype=np.float32)
    k_eff = max(1, min(k_swd_eff, N-1))

    t0 = time.time()
    for t in range(N):
        sims_t = sim[t]
        idxs = np.argpartition(sims_t, -(k_eff + 1))[-(k_eff + 1):]
        idxs = idxs[idxs != t]
        if idxs.size > 0:
            sv = sims_t[idxs]
            w  = compute_weights(sv, a, eps)
            diffs = pairwise_label_diff_scalar(ytr[t], ytr[idxs], mode=LABEL_DIFF_MODE)
            SWD[t] = float((w * sv * diffs).sum() / (w.sum() + eps))

        if (t + 1) % print_every == 0 or (t + 1) == N:
            dt = time.time() - t0
            print(f"    [SWD fold {fold_id}] {t+1}/{N} done ({dt:.1f}s)", flush=True)

    return SWD, Xn

def compute_rho_IA_given_SWD(Xtr_n, SWD, Xva, k_eff, a, eps):
    Xva_n = l2_normalize(Xva)
    sim_qt = cosine_sim_query(Xva_n, Xtr_n)  # (M,N)
    M = Xva.shape[0]
    rho = np.zeros(M, dtype=np.float32)
    IA  = np.zeros(M, dtype=np.float32)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k_eff)[-k_eff:]
        sk = sims[idxk]
        w  = compute_weights(sk, a, eps)
        rho[i] = float(w.mean())
        IA[i]  = float((w * SWD[idxk]).sum() / (w.sum() + eps))
    return rho, IA

# =========================
# 主流程
# =========================
def main():
    # 清理旧输出（如果你想续跑，注释掉）
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    # 预读 folds
    folds = []
    for f in range(1, n_folds+1):
        X_tr, y_tr, X_va, y_va, y_prob_va = load_fold_npz(f)
        folds.append((X_tr, y_tr, X_va, y_va, y_prob_va))

    total_jobs = len(a_list) * len(epsilon_list) * len(k_list)
    job_id = 0
    t_global = time.time()

    for a_weight in a_list:
        for eps in epsilon_list:
            print(f"\n=== START (a={a_weight}, eps={eps:.0e}) ===", flush=True)

            # 先算 SWD（并打印进度）
            swd_cache = []
            for fold_id, (X_tr, y_tr, X_va, y_va, y_prob_va) in enumerate(folds, start=1):
                k_swd_eff = min(max(int(0.1 * X_tr.shape[0]), 1), K_SWD_CAP)
                print(f"  -> computing SWD for fold {fold_id}, k_swd_eff={k_swd_eff}", flush=True)
                SWD, Xtr_n = compute_SWD_topk_with_progress(
                    X_tr, y_tr, k_swd_eff, a_weight, eps, fold_id=fold_id, print_every=500
                )
                swd_cache.append((SWD, Xtr_n, k_swd_eff))

            # 再对每个 k_ratio 输出一行结果
            for k_ratio in k_list:
                job_id += 1
                t_job = time.time()

                fold_cache = []
                all_rho, all_IA = [], []

                for idx, (X_tr, y_tr, X_va, y_va, y_prob_va) in enumerate(folds):
                    SWD, Xtr_n, k_swd_eff = swd_cache[idx]
                    k_eff = max(int(float(k_ratio) * X_tr.shape[0]), 1)
                    k_eff = min(k_eff, X_tr.shape[0])

                    rho_s, IA = compute_rho_IA_given_SWD(Xtr_n, SWD, X_va, k_eff, a_weight, eps)

                    fold_cache.append((rho_s, IA, y_va, y_prob_va))
                    all_rho.append(rho_s); all_IA.append(IA)

                all_rho = np.concatenate(all_rho)
                all_IA  = np.concatenate(all_IA)

                # baseline & 固定标签集合
                Y_all = np.concatenate([fc[2] for fc in fold_cache], axis=0)
                P_all = np.concatenate([fc[3] for fc in fold_cache], axis=0)
                ap_baseline = per_label_ap(Y_all, P_all)
                label_idx = np.array([j for j in range(Y_all.shape[1]) if not np.isnan(ap_baseline[j])], dtype=int)

                if label_idx.size == 0:
                    best_ap = np.nan
                    best_cov = np.nan
                    best_rho_th = np.nan
                    best_ia_th = np.nan
                else:
                    rho_cuts = np.quantile(all_rho, np.linspace(Q_LOW, Q_HIGH, GRID_SIZE))
                    IA_cuts  = np.quantile(all_IA,  np.linspace(Q_LOW, Q_HIGH, GRID_SIZE))

                    best_ap = -np.inf
                    best_cov = 0.0
                    best_rho_th = None
                    best_ia_th = None

                    for ia_th in IA_cuts:
                        for rho_th in rho_cuts:
                            s, cnt = 0.0, 0
                            cov_sum = 0.0
                            for (rho_s, IA, y_va, y_prob_va) in fold_cache:
                                in_m = (rho_s >= rho_th) & (IA <= ia_th)
                                n_in = int(in_m.sum())
                                if n_in < MIN_IN_SAMPLES:
                                    continue
                                cov_sum += n_in / len(rho_s)
                                ap_in = macro_auprc_with_baseline_fill(
                                    y_va[in_m], y_prob_va[in_m],
                                    ap_baseline, label_idx
                                )
                                s += ap_in
                                cnt += 1
                            if cnt > 0:
                                mean_ap = s / cnt
                                mean_cov = cov_sum / cnt
                                if mean_ap > best_ap:
                                    best_ap = mean_ap
                                    best_cov = mean_cov
                                    best_rho_th = float(rho_th)
                                    best_ia_th  = float(ia_th)

                    if not np.isfinite(best_ap):
                        best_ap = np.nan
                        best_cov = np.nan
                        best_rho_th = np.nan
                        best_ia_th = np.nan

                row = {
                    "job_id": job_id,
                    "job_total": total_jobs,
                    "a": int(a_weight),
                    "eps": float(eps),
                    "k_ratio": float(k_ratio),
                    "GRID_SIZE": int(GRID_SIZE),
                    "MIN_IN_SAMPLES": int(MIN_IN_SAMPLES),
                    "K_SWD_CAP": int(K_SWD_CAP),
                    "best_mean_AUPRC_in": float(best_ap) if np.isfinite(best_ap) else np.nan,
                    "best_mean_coverage_in": float(best_cov) if np.isfinite(best_cov) else np.nan,
                    "best_rho_th": best_rho_th,
                    "best_IA_th": best_ia_th,
                    "label_diff_mode": LABEL_DIFF_MODE,
                    "elapsed_sec_this_job": round(time.time() - t_job, 3),
                    "elapsed_min_total": round((time.time() - t_global) / 60.0, 2),
                }

                append_row_to_csv(row, OUT_CSV)

                print(
                    f"[{job_id}/{total_jobs}] a={a_weight:<5} eps={eps:.0e} k={k_ratio:>4.2f} | "
                    f"AUPRC_in={row['best_mean_AUPRC_in']:.4f} cov={row['best_mean_coverage_in']:.3f} | "
                    f"(rho_th={row['best_rho_th']:.4g}, IA_th={row['best_IA_th']:.4g}) | "
                    f"{row['elapsed_sec_this_job']}s",
                    flush=True
                )

    print("\n[SAVED]", OUT_CSV, flush=True)

if __name__ == "__main__":
    main()


=== START (a=10, eps=1e-03) ===
  -> computing SWD for fold 1, k_swd_eff=300
    [SWD fold 1] 500/3961 done (0.1s)
    [SWD fold 1] 1000/3961 done (0.1s)
    [SWD fold 1] 1500/3961 done (0.2s)
    [SWD fold 1] 2000/3961 done (0.3s)
    [SWD fold 1] 2500/3961 done (0.3s)
    [SWD fold 1] 3000/3961 done (0.4s)
    [SWD fold 1] 3500/3961 done (0.4s)
    [SWD fold 1] 3961/3961 done (0.5s)
  -> computing SWD for fold 2, k_swd_eff=300
    [SWD fold 2] 500/3961 done (0.1s)
    [SWD fold 2] 1000/3961 done (0.1s)
    [SWD fold 2] 1500/3961 done (0.2s)
    [SWD fold 2] 2000/3961 done (0.3s)
    [SWD fold 2] 2500/3961 done (0.3s)
    [SWD fold 2] 3000/3961 done (0.4s)
    [SWD fold 2] 3500/3961 done (0.5s)
    [SWD fold 2] 3961/3961 done (0.5s)
  -> computing SWD for fold 3, k_swd_eff=300
    [SWD fold 3] 500/3962 done (0.1s)
    [SWD fold 3] 1000/3962 done (0.1s)
    [SWD fold 3] 1500/3962 done (0.2s)
    [SWD fold 3] 2000/3962 done (0.3s)
    [SWD fold 3] 2500/3962 done (0.3s)
    [SWD fold 3]

In [ ]:
 a=10    eps=1e-03 k=0.05 | AUPRC_in=0.4604 cov=0.120 | (rho_th=0.007865, IA_th=0.5819) | 60.102s